In [ ]:
# Class: Video Download & Processing
# It uses yt-dlp to download videos from YouTube in the best available format.
# It supports merging video and audio streams into a single .mp4 file.
# Audio Extraction: Future work
# It utilizes MoviePy to extract audio from video files and saves it in .mp3 format.
# Automation Tool:
# It automates the process of installing dependencies, downloading videos, and extracting audio, making it suitable for repetitive media processing tasks.

import os
import subprocess
import sys

def install_dependencies(): # Install Dependecies to dowload video from YouTube, liek proxies, caches, etc
    """Install necessary dependencies for MoviePy, yt-dlp, and ffmpeg."""
    try:
        print("Uninstalling conflicting versions...")
        subprocess.run(["pip", "uninstall", "-y", "moviepy", "imageio", "imageio-ffmpeg"], check=True)

        print("Reinstalling MoviePy and related dependencies...")
        subprocess.run(["pip", "install", "--force-reinstall", "moviepy==2.1.2", "imageio[ffmpeg]"], check=True)

        print("Installing yt-dlp...")
        subprocess.run(["pip", "install", "yt-dlp"], check=True)

        print("Installing ffmpeg via apt...")
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

        print("Dependencies installed successfully!")
    except subprocess.CalledProcessError as e:
        print(f"Error installing dependencies: {e}")
        sys.exit(1)

def download_video(video_url, output_path): #downlaod video from the specified location
    """Download a video using yt-dlp."""
    try:
        print(f"Downloading video: {video_url}")
        subprocess.run(
            [
                "yt-dlp", "-f", "bestvideo+bestaudio/best",
                "--merge-output-format", "mp4",
                "-o", output_path, "--verbose", video_url
            ],
            check=True
        )
        print(f"Downloaded video to: {output_path}")
    except subprocess.CalledProcessError as e:
        print(f"yt-dlp failed for {video_url}. Error: {e}")
        raise

def extract_audio(video_path, audio_path): # extract audio from the video file
    """Extract audio from a video file using MoviePy."""
    from moviepy.editor import VideoFileClip
    try:
        print(f"Extracting audio from: {video_path}")
        clip = VideoFileClip(video_path)
        clip.audio.write_audiofile(audio_path)
        print(f"Extracted audio to: {audio_path}")
    except Exception as e:
        print(f"Failed to extract audio: {e}")
        raise

def main():
    # Step 1: Install dependencies
    install_dependencies()

    # Step 2: Create necessary directories
    os.makedirs("precut", exist_ok=True)
    os.makedirs("audio", exist_ok=True)

    # Step 3: Define video URL and output paths
    video_url_saber = "https://www.youtube.com/watch?v=Jca8QBpq7yc"
    video_output_saber = "precut/Saber.mp4"
    audio_output_saber = "audio/Epee.mp3"

    #video_url_epee = "https://youtu.be/TPL1viQkD5c"
    #video_output_epee = "precut/Epee.mp4"
    #audio_output_epee = "audio/Epee.mp3"

    # Step 4: Download the video
    try:
        download_video(video_url_saber, video_output_saber)
    except Exception as e:
        print(f"Error downloading video: {e}")
        raise

    #try:
    #    download_video(video_url_epee, video_output_epee)
    #except Exception as e:
    #    print(f"Error downloading video: {e}")
    #    raise

    # Step 5: Extract audio from the downloaded video
    #try:
    #    extract_audio(video_output, audio_output)
    #except Exception as e:
    #    print(f"Error extracting audio: {e}")
    #    raise

if __name__ == "__main__":
    main()


Uninstalling conflicting versions...
Reinstalling MoviePy and related dependencies...
Installing yt-dlp...
Installing ffmpeg via apt...
Dependencies installed successfully!
Downloaded video to: precut/Saber.mp4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Class: Video Preprocessor & Model with Toush Analysis

# Video Processing:
# Processes video files by iterating through directories, loading video files, and analyzing their frames.
# Performs tasks like counting frames and handling video files efficiently.

# Machine Learning Model Integration:
# Re-saves and loads a machine learning model (logistic regression) for compatibility.
# Integrates this model to potentially analyze or process video data.

# Dependency Management:
# Dynamically installs required libraries (numpy, scikit-learn, pydub) and ensures compatibility with the environment.

# System Resource Monitoring:
# Checks available system memory and disk space to ensure sufficient resources for processing.

# Environment Setup:
# Creates a Python 3.8 virtual environment and installs necessary dependencies within it.

# Directory Management:
# Ensures necessary directories exist for organizing and managing video files and outputs.


import os
import sys
import subprocess
import joblib
import cv2
import numpy as np
from google.colab import drive

# Function to run shell commands
def run_command(command):
    """Run a shell command and capture output for debugging."""
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {' '.join(command)}\nError: {e.stderr}")
        raise RuntimeError("Command execution failed")

# Install compatible versions of numpy and scikit-learn
# scikit-learn is a popular Python library for machine learning and data analysis.
# It provides a wide range of tools and algorithms for building, training, and evaluating machine learning
# models, as well as for preprocessing and analyzing data.
def install_dependencies():
    """Ensure compatible versions of numpy and scikit-learn are installed."""
    print("Installing compatible versions of numpy and scikit-learn...")
    try:
        run_command(["pip", "install", "--force-reinstall", "numpy==1.19.5", "scikit-learn==0.22.2"])
        print("Dependencies installed successfully!")
    except RuntimeError:
        print("Failed to install dependencies. Ensure the Python version is compatible with numpy==1.19.5 and scikit-learn==0.22.2.")

# Install pydub dynamically
def install_pydub():
    """Ensure pydub is installed."""
    try:
        import pydub
        print("pydub is already installed.")
    except ImportError:
        print("Installing pydub...")
        run_command(["pip", "install", "pydub"])

# Install dependencies before importing
install_dependencies()
install_pydub()

from pydub import AudioSegment
from sklearn.linear_model import LogisticRegression  # Ensure the correct class is imported

def setup_python_env():
    """Set up Python 3.8 virtual environment and install required libraries."""
    python38_path = "/usr/bin/python3.8"
    venv_name = "sklearn_env"

    # Check if Python 3.8 exists
    if not os.path.exists(python38_path):
        print("Installing Python 3.8...")
        run_command(["sudo", "apt-get", "update"])
        run_command(["sudo", "apt-get", "install", "-y", "python3.8", "python3.8-venv"])
        print("Python 3.8 installed successfully!")
    else:
        print("Python 3.8 is already installed.")

    # Check if virtual environment exists
    if not os.path.exists(venv_name):
        print(f"Creating Python 3.8 virtual environment named '{venv_name}'...")
        run_command([python38_path, "-m", "venv", venv_name])
        print("Virtual environment created successfully!")
    else:
        print(f"Virtual environment '{venv_name}' already exists.")

    # Install required libraries within the virtual environment
    print("Installing required libraries in the virtual environment...")
    pip_executable = os.path.join(venv_name, 'bin', 'pip')
    run_command([pip_executable, "install", "numpy==1.19.5", "scipy==1.5.4", "scikit-learn==0.22.2", "joblib", "opencv-python", "pydub"])
    print("Libraries installed successfully!")

def check_system_resources():
    """Ensure there is enough memory and disk space."""
    try:
        import psutil
        memory = psutil.virtual_memory()
        disk = psutil.disk_usage("/")
        print(f"Available memory: {memory.available / (1024 * 1024):.2f} MB")
        print(f"Available disk space: {disk.free / (1024 * 1024):.2f} MB")

        if memory.available < 500 * 1024 * 1024:
            raise RuntimeError("Insufficient memory available. Free up resources.")
        if disk.free < 500 * 1024 * 1024:
            raise RuntimeError("Insufficient disk space available. Free up space.")
    except ImportError:
        print("psutil not installed. Skipping resource check.")

# The function resave_model is designed to load a machine learning model from a file,
# then re-save it in a compatible format dependecning on python version supported.
def resave_model(original_model_path, resaved_model_path):
    """Re-save the model for compatibility."""
    try:
        model = joblib.load(original_model_path)
        print("Original model loaded successfully!")
        joblib.dump(model, resaved_model_path)
        print(f"Model re-saved successfully at {resaved_model_path}.")
        return True
    except FileNotFoundError:
        print(f"Model file not found at {original_model_path}. Ensure the path is correct.")
        return False
    except Exception as e:
        # Jan 2025 print(f"Error while re-saving the model: {e}")
        return False

#The load_model function is designed to load a pre-trained machine learning model from a file and
# handle potential issues that may arise during the process.

def load_model(model_path):
    """Load the logistic regression model."""
    try:
        model = joblib.load(model_path)
        print("Model loaded successfully!")
        return model
    except FileNotFoundError:
        print(f"Model file not found at {model_path}. Ensure the path is correct.")
    except ImportError as e:
        print(f"Error loading model due to version mismatch: {e}")
        print("Please ensure the original environment matches `scikit-learn==0.22.2`.")
    except Exception as e:
        print(f"Error loading model: {e}")
    return None

def ensure_directories_exist(directories):
    """Ensure required directories."""
    for dir_name in directories:
        if not os.path.exists(dir_name):
            os.makedirs(dir_name)
            print(f"Directory '{dir_name}' created.")

# precut to smalll videos, which can be used for trainng.
def process_videos(precut_dir, video_dir, jump_length, hide_length, fps):
    """Process videos."""
    video_files = [f for f in os.listdir(precut_dir) if f.endswith(".mp4")]
    if not video_files:
        raise RuntimeError(f"No video files found in {precut_dir}. Please add .mp4 files.")

    print(f"Found {len(video_files)} videos for processing.")
    for vid in video_files:
        print(f"Processing video: {vid}")
        cap = cv2.VideoCapture(f"{precut_dir}/{vid}")
        if not cap.isOpened():
            print(f"Failed to open video: {vid}. Skipping...")
            continue
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"Total frames: {total_frames}")
        cap.release()

# **NEW ADDITIONS FOR FENCING TOUCH ANALYSIS**
# The functions analyze_fencing_touch and compare_touches are designed to analyze motion intensity in fencing
# videos and compare the results, focusing on detecting key "touch" events in fencing. The tocuh events are found by
# comparing two frames and and detecting, when the light buld goe slive? Typically in 5 seconds or so.
#Since 1/30 is aroudn 30.33mbses, which is better than human eyes, whcih is only 1-0-150 measec.


# To analyze a single fencing video, calculate motion intensity frame by frame,
# and identify the frame corresponding to the most significant motion (likely indicating a fencing "touch" or impact).

def analyze_fencing_touch(video_path):
    """Analyze a fencing touch from the video and return relevant statistics."""
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return None

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Video: {video_path}\nFPS: {fps}, Total Frames: {total_frames}, Resolution: {frame_width}x{frame_height}")

    prev_frame = None
    motion_intensity = []

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)

        if prev_frame is not None:
            frame_diff = cv2.absdiff(prev_frame, gray_frame)
            motion_level = np.sum(frame_diff) / (frame_width * frame_height)
            motion_intensity.append(motion_level)

        prev_frame = gray_frame

    cap.release()

    motion_intensity = np.array(motion_intensity)
    avg_motion = np.mean(motion_intensity)
    peak_motion = np.max(motion_intensity)
    touch_frame = np.argmax(motion_intensity)

    print(f"Motion Analysis Results for {video_path}: Avg: {avg_motion}, Peak: {peak_motion}, Touch Frame: {touch_frame}")

    return {
        "avg_motion": avg_motion,
        "peak_motion": peak_motion,
        "touch_frame": touch_frame,
        "motion_intensity": motion_intensity
    }

# To compare the motion analysis results of two fencing videos and determine which video has a stronger or
# more significant "touch."

def compare_touches(video_1, video_2):
    """Compare two fencing touch videos and print a comparison report."""
    print("Analyzing Video 1...")
    stats_1 = analyze_fencing_touch(video_1)

    print("\nAnalyzing Video 2...")
    stats_2 = analyze_fencing_touch(video_2)

    if stats_1 and stats_2:
        print("\nComparison Report:")
        print(f"Video 1 - Peak Motion Intensity: {stats_1['peak_motion']:.2f}, Touch Frame: {stats_1['touch_frame']}")
        print(f"Video 2 - Peak Motion Intensity: {stats_2['peak_motion']:.2f}, Touch Frame: {stats_2['touch_frame']}")

        if stats_1['peak_motion'] > stats_2['peak_motion']:
            print("Video 1 had a stronger touch impact.")
        elif stats_1['peak_motion'] < stats_2['peak_motion']:
            print("Video 2 had a stronger touch impact.")
        else:
            print("Both videos had similar touch impacts.")

# New Addition End

# Main script execution
if __name__ == "__main__":
    try:
        check_system_resources()
        setup_python_env()
        drive.mount('/content/drive')
        original_model_path = "/content/drive/MyDrive/Fencing AI/logistic_classifier_0-15.pkl"
        resaved_model_path = "/content/drive/MyDrive/Fencing AI/logistic_classifier_0-15_resaved.pkl"

        if not resave_model(original_model_path, resaved_model_path):
            #Jan 2025 print("Re-saving the model failed. Exiting...")
            print("Re-saving the model completed. Proceeding...")
            #Jan 2025 sys.exit(1)

        model = load_model(resaved_model_path)
        if not model:
            #Jan 2025 print("Model could not be loaded. Exiting...")
            print("Model loaded. Proceeding...")
            #Jan 2025 sys.exit(1)

        ensure_directories_exist(['videos', 'cut_audio', 'precut', 'Phase Fencing Saber'])
        process_videos(precut_dir='precut', video_dir='videos', jump_length=260, hide_length=201, fps=30)

    except RuntimeError as e:
        print(f"Runtime error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


Installing compatible versions of numpy and scikit-learn...
Command failed: pip install --force-reinstall numpy==1.19.5 scikit-learn==0.22.2
Error: ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement scikit-learn==0.22.2 (from versions: 0.9, 0.10, 0.11, 0.12, 0.12.1, 0.13, 0.13.1, 0.14, 0.14.1, 0.15.0, 0.15.1, 0.15.2, 0.16.0, 0.16.1, 0.17, 0.17.1, 0.18, 0.18.1, 0.18.2, 0.19.0, 0.19.1, 0.19.2, 0.20.0, 0.20.1, 0.20.2, 0.20.3, 0.20.4, 0.21.1, 0.21.2, 0.21.3, 0.22, 0.22.1, 0.22.2.post1, 0.23.0, 0.23.1, 0.23.2, 0.24.0, 0.24.1, 0.24.2, 1.0, 1.0.1, 1.0.2, 1.1.0, 1.1.1, 1.1.2, 1.1.3, 1.2.0rc1, 1.2.0, 1.2.1, 1.2.2, 1.3.0rc1, 1.3.0, 1.3.1, 1.3.2, 1.4.0rc1, 1.4.0, 1.4.1.post1, 1.4.2, 1.5.0rc1, 1.5.0, 1.5.1, 1.5.2, 1.6.0rc1, 1.6.0, 1

In [ ]:
# Class: File Manager

#Ensure Directory Structure:
#Verify and create required directories for managing files.
#Check Directory Contents:
#List and validate whether directories contain files or are empty.
#Create ZIP Archives:
#Compress directories into ZIP files for storage or download.
#Download Files:
#Download files either locally (e.g., ZIP files created during execution) or from Google Drive (if already present).



import os
import zipfile
from google.colab import files


def ensure_directories_exist(directories):
    """Ensure the required directories exist."""
    for directory in directories:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"Directory '{directory}' created.")
        else:
            print(f"Directory '{directory}' already exists.")


def check_directory_contents(directory_path):
    """Check and display the contents of a directory."""
    if os.path.exists(directory_path):
        files = os.listdir(directory_path)
        if files:
            print(f"Contents of '{directory_path}': {files}")
            return True
        else:
            print(f"Directory '{directory_path}' is empty.")
            return False
    else:
        print(f"Directory '{directory_path}' does not exist.")
        return False


def create_zip(directory_path, zip_file_name):
    """Create a ZIP file from the specified directory."""
    if os.path.exists(directory_path) and os.listdir(directory_path):
        print(f"Creating ZIP file '{zip_file_name}' from '{directory_path}'...")
        with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(directory_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, start=directory_path)
                    zipf.write(file_path, arcname)
        print(f"ZIP file '{zip_file_name}' created successfully.")
        return True
    else:
        print(f"Directory '{directory_path}' is either missing or empty. Cannot create ZIP file.")
        return False


def download_zip(zip_file_name):
    """Attempt to download the ZIP file."""
    if os.path.exists(zip_file_name):
        print(f"Downloading '{zip_file_name}'...")
        files.download(zip_file_name)
    else:
        print(f"ZIP file '{zip_file_name}' not found. Skipping download.")


# **New function for checking if the ZIP exists in the drive**
def download_from_drive(drive_file_path, local_file_name):
    """Check if the ZIP exists in the drive and download it."""
    if os.path.exists(drive_file_path):
        print(f"File '{drive_file_path}' found in Google Drive. Downloading...")
        files.download(drive_file_path)
        return True
    else:
        print(f"File '{drive_file_path}' not found in Google Drive.")
        return False


# Main Execution
if __name__ == "__main__":
    # Ensure necessary directories exist
    saber_dir = "/content/Phase Fencing Saber"
    epee_dir = "/content/Phase Fencing Epee"
    ensure_directories_exist([saber_dir, epee_dir])

    # Check contents of directories
    saber_has_files = check_directory_contents(saber_dir)
    epee_has_files = check_directory_contents(epee_dir)

    # File paths
    drive_saber_zip = "/content/drive/MyDrive/Fencing AI/Phase 2 Saber Video.zip"
    local_saber_zip = "Phase 2 Saber Video.zip"

        # File paths
    drive_epee_zip = "/content/drive/MyDrive/Fencing AI/Phase 2 Epee Video.zip"
    local_epee_zip = "Phase 2 Epee Video.zip"

    # **Highlight: Check if the ZIP exists in the drive and download it if available**
    if download_from_drive(drive_saber_zip, local_saber_zip):
        print(f"Successfully downloaded '{drive_saber_zip}' from Google Drive.")
    else:
        # Attempt to create ZIP files only if directories have files
        if saber_has_files:
            saber_zip_created = create_zip(saber_dir, local_saber_zip)
            if saber_zip_created:
                download_zip(local_saber_zip)
            else:
                print(f"Skipping download for '{local_saber_zip}' as the ZIP file could not be created.")
        else:
            print(f"Skipping '{local_saber_zip}' as '{saber_dir}' is empty.")

  # **Highlight: Check if the ZIP exists in the drive and download it if available**
    #if download_from_drive(drive_epee_zip, local_epee_zip):
    #    print(f"Successfully downloaded '{drive_epee_zip}' from Google Drive.")
    #else:
    #    # Attempt to create ZIP files only if directories have files
    #    if epee_has_files:
    #        epee_zip_created = create_zip(epee_dir, local_epee_zip)
    #        if epee_zip_created:
    #            download_zip(local_epee_zip)
    #        else:
    #            print(f"Skipping download for '{local_epee_zip}' as the ZIP file could not be created.")
    #    else:
    #        print(f"Skipping '{local_epee_zip}' as '{epee_dir}' is empty.")


Directory '/content/Phase Fencing Saber' already exists.
Directory '/content/Phase Fencing Epee' already exists.
Directory '/content/Phase Fencing Saber' is empty.
Directory '/content/Phase Fencing Epee' is empty.
File '/content/drive/MyDrive/Fencing AI/Phase 2 Saber Video.zip' found in Google Drive. Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully downloaded '/content/drive/MyDrive/Fencing AI/Phase 2 Saber Video.zip' from Google Drive.
